In [1]:
!pip install -U sentence-transformers

  Using cached torch-2.10.0-cp314-cp314-macosx_14_0_arm64.whl.metadata (31 kB)
  Using cached setuptools-82.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached markupsafe-3.0.3-cp314-cp314-macosx_11_0_arm64.whl.metadata (2.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 5.6 MB/s  0:00:01m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 7.3 MB/s  0:00:00 eta 0:00:01
Using cached torch-2.10.0-cp314-cp314-macosx_14_0_arm64.whl (79.5 MB)
Using cached networkx-3.6.1-py3-none-any.whl (2.1 MB)
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
Using cached markupsafe-3.0.3-cp314-cp314-macosx_11_0_arm64.whl 

In [1]:
# Load RecruitView dataset and parse transcripts by timestamp (one row per participant)
import pandas as pd
from datasets import load_dataset

dataset = load_dataset("AI4A-lab/RecruitView")
train = dataset["train"]

def parse_transcript(transcript_str):
    """Parse '[00:01 - 00:11] text' lines into list of (timestamp, text)."""
    if not transcript_str or not transcript_str.strip():
        return []
    segments = []
    for line in transcript_str.strip().split("\n"):
        line = line.strip()
        if not line:
            continue
        if line.startswith("[") and "]" in line:
            idx = line.index("]")
            timestamp = line[1:idx].strip()
            text = line[idx + 1 :].strip()
            if text:
                segments.append((timestamp, text))
        else:
            segments.append(("", line))
    return segments

# Use personality_score from dataset if present, else placeholder
personality_col = None
for col in ("personality_score", "overall_personality", "personality"):
    if col in train.column_names:
        personality_col = col
        break

# Build table: one row per participant; transcript_segments = { "0:01 - 0:11": "words", ... }
rows = []
for participant_id in range(len(train)):
    transcript_str = train["transcript"][participant_id]
    score = train[personality_col][participant_id] if personality_col else None
    segments = parse_transcript(transcript_str)
    transcript_segments = {ts: text for ts, text in segments}
    rows.append({
        "participant_id": participant_id,
        "transcript_segments": transcript_segments,
        "personality_score": score,
    })

table = pd.DataFrame(rows)
print(f"Loaded {len(train)} participants (one row each).")
print("Table columns:", list(table.columns))
print("Example transcript_segments (first participant):", table["transcript_segments"].iloc[0])
table.head()

/Users/harishdukkipati/Downloads/CSCI-566-Course-Project-DeepPrep-AI/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 2011 participants → 8805 segments.
Table columns: ['participant_id', 'timestamp', 'transcript', 'personality_score']
   participant_id      timestamp  \
0               0  00:01 - 00:11   
1               0  00:12 - 00:18   
2               0  00:18 - 00:30   
3               0  00:30 - 00:41   
4               1  00:00 - 00:05   

                                          transcript  personality_score  
0  Hello everyone, this is Mathurima Roy. I am a ...          -0.028877  
1  I am a tech enthusiast and I love to explore t...          -0.028877  
2  I also have a sheer interest in exploring the ...          -0.028877  
3  My hobbies include dancing, reading, I am an a...          -0.028877  
4  I'm Darshita Singh from Aligarh, UP. I am stud...          -0.064224  


In [2]:
# Embed segment transcripts: one 2D array per participant (n_segments, embed_dim)
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# One list of segment texts per participant (order preserved from dict)
segment_texts_per_participant = [list(row["transcript_segments"].values()) for _, row in table.iterrows()]
# Flatten to encode all segments in one batch
all_segment_texts = [t for segs in segment_texts_per_participant for t in segs]
embeddings_flat = model.encode(all_segment_texts, show_progress_bar=True)

# Split back into 2D arrays: one (n_segments, embed_dim) per participant
sizes = [len(segs) for segs in segment_texts_per_participant]
splits = np.cumsum(sizes)[:-1]
table["transcript_embeddings"] = np.split(embeddings_flat, splits)

print(f"Total segments: {len(all_segment_texts)}. Embedding dim: {embeddings_flat.shape[1]}.")
print("Per-participant shapes (n_segments, embed_dim):", [e.shape for e in table["transcript_embeddings"].iloc[:3]])
table.head()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2175.94it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 276/276 [00:06<00:00, 42.24it/s] 

Embeddings shape: (8805, 384)
Table columns: ['participant_id', 'timestamp', 'transcript', 'personality_score', 'transcript_embeddings']


,participant_id,timestamp,transcript,personality_score,transcript_embeddings
0,0,00:01 - 00:11,"Hello everyone, this is Mathurima Roy. I am a ...",-0.028877,"[-0.022972569, -0.027451137, 0.010178116, 0.03..."
1,0,00:12 - 00:18,I am a tech enthusiast and I love to explore t...,-0.028877,"[0.004986396, -0.14033228, 0.01638942, 0.00907..."
2,0,00:18 - 00:30,I also have a sheer interest in exploring the ...,-0.028877,"[-0.0196175, -0.070486836, 0.018671889, -0.030..."
3,0,00:30 - 00:41,"My hobbies include dancing, reading, I am an a...",-0.028877,"[0.03294871, -0.097717166, 0.057788976, -0.006..."
4,1,00:00 - 00:05,"I'm Darshita Singh from Aligarh, UP. I am stud...",-0.064224,"[-0.014245266, -0.025699891, 0.06324625, 0.068..."
